In [14]:
%load_ext autoreload
%autoreload 2

import sys
import os
import pandas as pd
import time
from sb3_contrib import MaskablePPO
from sb3_contrib.common.wrappers import ActionMasker
from sb3_contrib.common.maskable.utils import get_action_masks

# Get the absolute path of the parent directory
parent_dir = os.path.abspath('..')

# Add the parent directory to the system path
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

from jssp_env import parser
from jssp_env import rl_env_flat

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
def train_generalization_10x10(model_name: str, entropy: float = 0.0) -> None:
    """
    Train the parameterized model.
    """
    env = rl_env_flat.FlatJSSPEnv(nb_jobs=10, nb_machines=10)

    env = ActionMasker(
        env,
        lambda e: e.get_wrapper_attr('job_next_task_index') < e.get_wrapper_attr('nb_machines')
    )

    # PPO Agent:
    model = MaskablePPO(
        "MlpPolicy",
        env,
        ent_coef=entropy,
        learning_rate=3e-4,
        policy_kwargs=dict(net_arch=dict(pi=[256, 256], vf=[256, 256])),
        verbose=1,
        tensorboard_log="./tensorboard_logs/"
    )

    print(f"Training {model_name}")
    model.learn(
        total_timesteps=300000,
        tb_log_name=f"training_10x10_ent_{entropy}"
    )

    model.save(f"saved_models/{model_name}")
    print(f"Model {model_name} saved")

    env.close()

def test_on_ft10(model_name: str) -> dict:
    """
    Test the parameterized model on the "Fisher and Thompson 10 x 10 instance"
    """
    instance_ft10 = parser.parse_jssp_instance_from_file("../data/ft10.txt")
    
    env = rl_env_flat.FlatJSSPEnv(instance=instance_ft10)
    env = ActionMasker(
        env,
        lambda e: e.get_wrapper_attr('job_next_task_index') < e.get_wrapper_attr('nb_machines')
    )
    
    model = MaskablePPO.load(f"saved_models/{model_name}")
    
    obs, info = env.reset()
    terminated = False
    truncated = False

    # Warm-up for initialization
    action_masks = get_action_masks(env)
    model.predict(obs, action_masks=action_masks, deterministic=True)
    
    total_score = 0
    
    while not (terminated or truncated):
        action_masks = get_action_masks(env)
        
        start_time = time.perf_counter()
        action, _states = model.predict(obs, action_masks=action_masks, deterministic=True)
        end_time = time.perf_counter()
        
        obs, reward, terminated, truncated, info = env.step(action)
        total_score += reward

    obtained_makespan = max(env.get_wrapper_attr('machine_available_time'))

    inference_time = (end_time - start_time) * 1000

    env.close()

    return {
        "makespan": obtained_makespan,
        "inference_time_ms": inference_time
    }

In [ ]:
data = []
counter = 1

# Loop over the two entropy values we want to test
for entropy_val in [0.0, 0.05]:
    # For each entropy, perform 5 different training runs
    for i in range(1, 6):
        # Generate a unique model name
        model_name = f"maskable_ppo_10x10_ent_{entropy_val}_run_{i}"
        
        train_generalization_10x10(model_name=model_name, entropy=entropy_val)
        
        makespan = test_on_ft10(model_name=model_name)["makespan"]
        
        data.append({
            "Training": f"Training {counter}",
            "Entropy": entropy_val,
            "Makespan": makespan
        })
        counter += 1

# Direct DataFrame creation from the list of dictionaries
df_results = pd.DataFrame(data)
df_results.set_index("Training", inplace=True)

display(df_results)

In [ ]:
mask_no_entropy = df_results["Entropy"] == 0
mean_no_entropy = df_results[mask_no_entropy]["Makespan"].mean()
std_no_entropy = df_results[mask_no_entropy]["Makespan"].std()

mask_with_entropy = df_results["Entropy"] == 0.05
mean_with_entropy = df_results[mask_with_entropy]["Makespan"].mean()
std_with_entropy = df_results[mask_with_entropy]["Makespan"].std()

print("The optimal solution for ft10 is 930 time units")
print(f"Our model without entropy found an average solution of {mean_no_entropy:.2f} +/- {std_no_entropy:.2f} time units")
print(f"Our model with an entropy of 0.05 found an average solution of {mean_with_entropy:.2f} +/- {std_with_entropy:.2f} time units")